# Data preparation for Llama
Stage 2 & 3

## A Overview

In [ ]:
from pathlib import Path
import csv
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
import evaluate

import torch

import configuration

from src import setup, pipeline

## B. Configs

In [ ]:
strategy = "isolated"  # "isolated" or "crossed"
times = 2
models_path = Path("..") / "models"
datasets_path = Path("..") / "data" / "splitted" / "stage_3"
tokens_path = Path("..") / "tokens" / "stage_3" / strategy 

safety_net_path = datasets_path / strategy
safety_net_path.parent.mkdir(parents=True, exist_ok=True)

# BERT
bert_model_name = "bert-base-uncased"
bert_model_path = (
    Path("..")
    / "models"
    / "stage_1"
    / f"{strategy}"
    / "BERT"
    / "w12890_o552597"
    / "1.0"
)
bert_tokenized_path = tokens_path / "BERT"/ "no_informative"
bert_tokenized_path.mkdir(parents=True, exist_ok=True)

# LLaMA
llama_model_name = "meta-llama/Llama-3.2-3B"
llama_model_path = (
    Path("..")
    / "models"
    / "stage_2"
    / f"{strategy}"
    / f"{times}_times"
    / "LLama_3_2_3B"
)

llama_tokenized_path = tokens_path / "LLaMA"/ "no_informative"
llama_tokenized_path.mkdir(parents=True, exist_ok=True)


device = setup.setup_device_with_seeds()

GPU: NVIDIA A100-SXM4-80GB
Memory allocated: 12.7 GB
Memory cached: 57.5 GB
Using device: cuda


In [ ]:
df_no_informative = pd.read_csv(datasets_path / "no_informative.csv")
df_humanitarian = pd.read_csv(datasets_path / "humanitarian.csv")
print(df_no_informative.shape)
print(df_humanitarian.shape)

df_no_informative["informative"].value_counts()

(54796, 5)
(84189, 5)


## C. Safety Net Data

## Stage 1

In [ ]:
bert_predictions, bert_confidences = pipeline.stage_1_bert_binary(df_no_informative, bert_model_name, bert_model_path, bert_tokenized_path, device)

In [ ]:
df_no_informative["stage_1_predicted"] = bert_predictions
df_no_informative["stage_1_confidence"] = bert_confidences

df_bert_failed_predictions = df_no_informative[
    df_no_informative["informative"] != df_no_informative["stage_1_predicted"]
]
print(df_bert_failed_predictions.shape)
df_bert_failed_predictions.head()

(23112, 7)


,uid,tweet_text,informative,humanitarian_label,subset,stage_1_predicted,stage_1_confidence
3,86175,RT keyetv \RT TVsJordanSteele: #HurricanePatri...,False,NaN,disaster,1,0.677521
4,229637,RT @HighvTweet: Empire State Buildings lit up ...,False,not_humanitarian,disaster,1,0.898826
5,187252,@mikeseidel just segued to a commercial on @we...,False,NaN,disaster,1,0.792914
6,22046,BREAKING&gt;#WhiteHouse Says #USA WILLNegotiat...,False,NaN,disaster,1,0.529898
11,11068,RT NotExplained: The only known image of infam...,False,NaN,disaster,1,0.662041


## Stage 2

In [ ]:
llama_predictions, llama_confidences = pipeline.stage_2_llama_binary(
    df_bert_failed_predictions,
    llama_model_name,
    llama_model_path,
    llama_tokenized_path,
    device,
)

In [ ]:
df_bert_failed_predictions["stage_2_predicted"] = llama_predictions
df_bert_failed_predictions["stage_2_confidence"] = llama_confidences

df_safety_net = df_bert_failed_predictions[
    df_bert_failed_predictions["informative"]
    != df_bert_failed_predictions["stage_2_predicted"]
]
print(df_safety_net.shape)
df_safety_net['humanitarian_label'] = "not_humanitarian"
df_safety_net.head()

(679, 9)


/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,uid,tweet_text,informative,humanitarian_label,subset,stage_1_predicted,stage_1_confidence,stage_2_predicted,stage_2_confidence
90,114227,RT @AucklandCity_FC: VANUATU | @vanuafoot @HIR...,False,not_humanitarian,disaster,1,0.891030,1,0.613281
381,148254,"With a CERF allocation of almost $1.7 million,...",False,not_humanitarian,disaster,1,0.727841,1,0.819336
394,186049,Starting tmrw we will take pre-orders 4 our Ti...,False,not_humanitarian,disaster,1,0.834534,1,0.656738
623,39617,What is the risk assessment for people in La P...,False,not_humanitarian,disaster,1,0.657170,1,0.918457
630,38645,We persist to send message for me but we never...,False,not_humanitarian,disaster,1,0.754758,1,0.991211


## D. Merge

In [58]:
n = min(int(df_humanitarian.shape[0] * 0.1), df_safety_net.shape[0])

df_safety_net = df_safety_net.sample(n=n, random_state=setup.RANDOM_SEED).to_csv(
    safety_net_path / "safety_net.csv",
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
)

In [59]:
df_stage_3 = pd.concat([df_safety_net, df_humanitarian], ignore_index=True)
df_stage_3.to_csv(
    datasets_path / "stage_3_dataset.csv", index=False, quoting=csv.QUOTE_NONNUMERIC
)

In [60]:
df_stage_3.shape

(84189, 5)

## Splitting

80:10:10

In [ ]:
df_train, df_validation = train_test_split(
    df_stage_3, test_size=0.2, random_state=setup.RANDOM_SEED
)
df_validation, df_test = train_test_split(
    df_validation, test_size=0.5, random_state=setup.RANDOM_SEED
)
df_train.to_csv(
    datasets_path / strategy / "train.csv",
    index=False,
    quoting=csv.QUOTE_NONNUMERIC
)
df_validation.to_csv(
    datasets_path / strategy / "validation.csv",
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
)
df_test.to_csv(
    datasets_path / strategy / "test.csv",
    index=False,
    quoting=csv.QUOTE_NONNUMERIC
)

In [ ]:
print(df_train.shape)
print(df_validation.shape)
print(df_test.shape)